# README

- **Author**: 楊喬茵（AI協助：Claude Code，Claude Opus 5.5）
- **Created At**: 2026-11-16
- **Last Modified At**: 2026-11-16

---

## What does this file do?

- 補 280 位候選人（臺北市議員 98、高雄市議員 101、全國縣市長 81）的 Facebook、Instagram、Threads 帳號。觀察單位是「候選人 × 平台」，以 `candidate_key` 連結，不以姓名跨人合併。
- 依楊喬茵 2026-11-16 指示，**直接更新原檔** `candidate_social_links_scope_20261114.csv`（不另建新版寬表）：只補狀態為 `not_located` 的格子，另加 `Facebook_note`、`Instagram_note`、`Threads_note` 標註確認層級與疑慮；既有連結與狀態不改，只在第三方目錄列出同一帳號時於 `*_note` 加註。
- 新來源由 [collect_social_links_20261116.py](collect_social_links_20261116.py) 蒐集：串仔 tshng-a 候選人頁（民進黨支持者整理，非官方）、DuckDuckGo 一般搜尋結果頁，以及搜尋找到、標題含候選人姓名的 Linktree 類連結頁。只用一般瀏覽器 User-Agent，未登入，未開啟 Facebook／Instagram／Threads 帳號頁（登入牆）。
- **確認層級（由強到弱）**：
  1. `third_party_directory_listed`：串仔候選人頁（已讀全頁、姓名縣市職位相符）列出該帳號。非官方，目標帳號未開啟。
  2. `candidate_link_hub_listed`：標題含候選人姓名的 Linktree 類頁面（已讀全頁）列出該帳號。連結頁本身的歸屬仍靠姓名判斷。
  3. `uncertain_search_profile_role_match`：**不確定**。搜尋結果中，平台自己的帳號頁標題含姓名，標題或簡介摘要含參選、職務或選區字詞；依 FINDING-RULES「網路資料與新聞須讀完全文」，摘要只算線索。
  4. `uncertain_search_profile_name_only`：**不確定**。只有帳號名稱含姓名，可能是同名他人。
- 帳號名稱（標題中平台分隔符號前的文字）必須含候選人姓名；名稱含後援會、粉絲、新聞等字詞的頁面記為 `excluded_fan_or_media_page`。摘要提到其他縣市而未提候選人所屬縣市者記為 `excluded_other_region`。兩者都不填入。
- 貼文結果的職務字詞常描述他人，因此貼文最多只算「僅名稱相符」。只有名稱相符且同名帳號超過一個時，不猜測，狀態記為 `uncertain_same_name_multiple`，urls 留空。
- 每格只填最強層級的帳號；較弱層級與排除的觀察留在證據表。
- 空值仍代表「本輪未找到」，不是「沒有帳號」。

---

## What does this file take?

- **Source Data Sets** (one entry per actual file):
  1. `data/processed/candidate_social_links_scope_20261114.csv`
     - Input key: `scope`
     - Description: 280 列寬表，三平台的 urls（JSON 字串陣列）與狀態。
     - Documentation: `data/processed/social_links_scope_20261114_documentation.ipynb`
     - SHA-256: `ac9dab6e14504395106de57eb9d2861dc029e400a743448ec42e756098d68e89`
  2. `data/raw/social_links_20261116/request_log.json`
     - Input key: `request_log`
     - Description: 本輪每次請求的網址、階段、候選人、平台、查詢字串、狀態、抓取時間、保存路徑與 hash。
     - Documentation: `data/raw/social_links_20261116/file_manifest.json`（逐檔 hash）；原始回應不改寫。
     - SHA-256: `954783d5f77d5b0ed840b0616470fa4368df58b9d840761b7498ecab6602efa4`
  3. `data/raw/social_links_20261116/file_manifest.json`
     - Input key: `manifest`
     - Description: 本輪保存的原始回應逐檔 SHA-256；讀取前逐檔核對。
     - SHA-256: `90816ba6beeddbf57fa41d059611b7c49614e30731cbdf955458d1da7157305f`

---

## What does this file output?

- `data/processed/candidate_social_links_scope_20261114.csv`（就地更新）
  - Output key: `scope_new`
  - Description: 280 列寬表；原 10 欄不變，另加 `Facebook_note`、`Instagram_note`、`Threads_note` 說明確認層級或疑慮。更新前版本在 git commit `4529580`（SHA-256 `ac9dab6e14504395106de57eb9d2861dc029e400a743448ec42e756098d68e89`），重跑本筆記本前需先取回該版本，否則輸入 hash 檢查會停止。
  - Role: collection / evidence（帳號入口清單，不是貼文資料）
  - Expected schema and row-count rules: 280 列、`candidate_key` 唯一且與輸入相同；非 `not_located` 的既有格子 urls 與 status 不變。
- `data/processed/social_link_evidence_added_20261116.csv`
  - Output key: `evidence`
  - Description: 本輪每一筆帳號觀察（含較弱層級與排除者）：候選人、平台、帳號網址、層級、來源網址、保存檔與 hash、標題與摘要原文。
  - Expected schema and row-count rules: 每列（candidate_key, platform, account_url, source_url）唯一；保存檔 hash 與 manifest 相符；標題原文確實出現在保存檔中。
- `data/processed/social_link_missing_summary_20261116.csv`
  - Output key: `summary`
  - Description: 職位 × 平台的本輪前後缺漏與各層級格數。

Documentation: 本筆記本（Data Artifacts 列登記於 `PROJECT.md`）。Verification status / SHA-256 見執行輸出。All paths are relative to the project root, not the operating-system root.


In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual


In [2]:
import json, re, urllib.parse, collections
from bs4 import BeautifulSoup

INPUTS = {"scope": "data/processed/candidate_social_links_scope_20261114.csv",
          "request_log": "data/raw/social_links_20261116/request_log.json",
          "manifest": "data/raw/social_links_20261116/file_manifest.json"}
EXPECTED_INPUT_SHA256 = {"scope": "ac9dab6e14504395106de57eb9d2861dc029e400a743448ec42e756098d68e89",
                         "request_log": "954783d5f77d5b0ed840b0616470fa4368df58b9d840761b7498ecab6602efa4",
                         "manifest": "90816ba6beeddbf57fa41d059611b7c49614e30731cbdf955458d1da7157305f"}
OUTPUTS = {"scope_new": "data/processed/candidate_social_links_scope_20261114.csv",  # in-place update
           "evidence": "data/processed/social_link_evidence_added_20261116.csv",
           "summary": "data/processed/social_link_missing_summary_20261116.csv"}
INPUT_OPTIONS = {"scope": {"dtype": str, "encoding": "utf-8-sig", "keep_default_na": False}}
OUTPUT_READ_OPTIONS = {k: {"dtype": str, "encoding": "utf-8-sig", "keep_default_na": False} for k in OUTPUTS}

input_paths = {k: project_file(p, Path(p).parts[1]) for k, p in INPUTS.items()}
output_paths = {k: project_file(p, "processed", must_exist=False) for k, p in OUTPUTS.items()}
assert len(set(output_paths.values())) == len(output_paths)
# The wide table is updated in place by explicit student instruction (2026-11-16); every other
# output must be new and distinct from the inputs.
assert output_paths["scope_new"] == input_paths["scope"]
others = {k: p for k, p in output_paths.items() if k != "scope_new"}
assert not set(input_paths.values()) & set(others.values())
existing = [str(p) for p in others.values() if p.exists()]
if existing:
    raise FileExistsError(f"Review existing outputs before replacing them: {existing}")
input_fingerprints = {k: verify_input_sha256(p, EXPECTED_INPUT_SHA256[k]) for k, p in input_paths.items()}

scope = pd.read_csv(input_paths["scope"], **INPUT_OPTIONS["scope"])
log = json.loads(input_paths["request_log"].read_text(encoding="utf-8"))
manifest = json.loads(input_paths["manifest"].read_text(encoding="utf-8"))
assert len(scope) == 280 and scope["candidate_key"].is_unique

# Every saved raw response must match its manifest hash before it is parsed.
raw_root = PROJECT_ROOT / "data/raw/social_links_20261116"
man = {m["path"]: m["sha256"] for m in manifest}
for rec in log:
    if rec["path"]:
        p = project_file(rec["path"], "raw")
        assert p.parent == raw_root.resolve() and man[rec["path"]] == rec["sha256"] == sha256_file(p), rec["path"]
print({"requests": len(log), "saved_files": len(man),
       "by_stage_status": collections.Counter((r["stage"], str(r["status"])) for r in log)})


{'requests': 712, 'saved_files': 711, 'by_stage_status': Counter({('ddg_search', '200'): 464, ('tshng_page', '200'): 238, ('link_hub', '200'): 6, ('tshng_sitemap', '200'): 2, ('ddg_search', 'error:URLError'): 1, ('ddg_search', '202'): 1})}


In [3]:
PLATFORMS = ["Facebook", "Instagram", "Threads"]
COUNTIES = ["臺北市", "新北市", "桃園市", "臺中市", "臺南市", "高雄市", "基隆市", "新竹市", "嘉義市", "新竹縣",
            "苗栗縣", "彰化縣", "南投縣", "雲林縣", "嘉義縣", "屏東縣", "宜蘭縣", "花蓮縣", "臺東縣", "澎湖縣",
            "金門縣", "連江縣"]

def core_name(name):
    return re.sub(r"[^㐀-鿿]", "", name)

def county_variants(c):
    short = c[:2]
    return {c, c.replace("臺", "台"), short, short.replace("臺", "台")}

ROLE_WORDS = ["議員", "參選", "候選人", "市長", "縣長", "立委", "立法委員", "服務處", "選區", "議會",
              "民進黨", "國民黨", "民眾黨", "時代力量", "基進", "綠黨", "社民黨", "里長", "競選"]

def read_raw(rec):
    return (PROJECT_ROOT / rec["path"]).read_bytes().decode("utf-8", "replace")

def normalize(url):
    """Return (platform, canonical account URL) or None for non-account URLs."""
    u = urllib.parse.urlparse(url.strip())
    host = u.netloc.lower().split(":")[0]
    host = host[4:] if host.startswith("www.") else host
    host = host[2:] if host.startswith("m.") else host
    parts = [p for p in u.path.split("/") if p]
    if host in ("facebook.com", "zh-tw.facebook.com", "web.facebook.com", "fb.com"):
        if not parts:
            return None
        if parts[0] == "profile.php":
            q = urllib.parse.parse_qs(u.query).get("id")
            return ("Facebook", f"https://www.facebook.com/profile.php?id={q[0]}") if q else None
        if parts[0] == "people" and len(parts) >= 3:
            return ("Facebook", f"https://www.facebook.com/{parts[2]}")
        if parts[0] == "p" and len(parts) >= 2:
            return ("Facebook", f"https://www.facebook.com/p/{parts[1]}")
        if parts[0].lower() in {"share", "sharer", "sharer.php", "groups", "events", "watch", "reel", "story.php",
                                "permalink.php", "photo.php", "photo", "hashtag", "login", "plugins", "dialog",
                                "tr", "pages", "public", "search", "help", "policies", "ads", "media", "video.php",
                                "l.php", "home.php", "marketplace", "gaming", "notes"}:
            return None
        return ("Facebook", f"https://www.facebook.com/{parts[0]}")
    if host == "instagram.com":
        if not parts or parts[0].lower() in {"p", "reel", "reels", "explore", "stories", "accounts", "tv", "about",
                                              "legal", "direct", "web", "developer"}:
            return None
        return ("Instagram", f"https://www.instagram.com/{parts[0].lower()}")
    if host in ("threads.net", "threads.com"):
        if not parts or not parts[0].startswith("@"):
            return None
        return ("Threads", f"https://www.threads.com/{parts[0].lower()}")
    return None

def is_profile_url(url):
    u = urllib.parse.urlparse(url)
    parts = [p for p in u.path.split("/") if p]
    if "facebook" in u.netloc and parts and parts[0] in ("people", "p", "profile.php"):
        return len(parts) <= 3
    return len(parts) <= 1

FAN_WORDS = ["後援會", "粉絲", "支持者", "新聞", "傳媒", "應援", "鐵粉", "討論", "宣傳站", "粉專分享"]

def account_label(title):
    """The account/page name: title text before the platform separator."""
    return re.split(r" - Facebook| \(@| • | on Instagram| on Threads| \| Facebook| - ", title, maxsplit=1)[0].strip()

key_by = {}
for _, r in scope.iterrows():
    kind = "議員" if "議員" in r["office_type"] else "長"
    key_by.setdefault((core_name(r["candidate_name"]), r["county_city"], kind), []).append(r["candidate_key"])
assert all(len(v) == 1 for v in key_by.values()), "name+county+office must identify one candidate"
scope_by_key = scope.set_index("candidate_key")

obs = []  # evidence observations

# 1. 串仔 directory pages (full page read programmatically; dedicated account links only).
TITLE = re.compile(r"^([^｜]+?)(?: [A-Z、]+ 社群帳號)?｜(.+?)候選人 2026｜串仔$")
tshng_matched = 0
for rec in [r for r in log if r["stage"] == "tshng_page" and r["status"] == 200]:
    html = read_raw(rec)
    soup = BeautifulSoup(html, "html.parser")
    m = TITLE.match(soup.title.get_text(strip=True) if soup.title else "")
    if not m:
        continue
    name, role = m.group(1).strip(), m.group(2)
    county = next((c for c in COUNTIES if role.startswith(c)), None)
    if county is None or not (role.endswith("議員") or role.endswith("市長") or role.endswith("縣長")):
        continue  # e.g. 村里長 pages are outside scope
    kind = "議員" if role.endswith("議員") else "長"
    keys = key_by.get((core_name(name), county, kind))
    if not keys:
        continue
    tshng_matched += 1
    for a in soup.find_all("a", href=True):
        n = normalize(a["href"])
        if n:
            obs.append(dict(candidate_key=keys[0], platform=n[0], account_url=n[1], tier="third_party_directory_listed",
                            source_url=rec["url"], source_file=rec["path"], source_sha256=rec["sha256"],
                            source_title=soup.title.get_text(strip=True), source_excerpt=a.get_text(" ", strip=True)[:200],
                            query=""))
print("串仔 pages matched to scope candidates:", tshng_matched)

# 2. DuckDuckGo results: platform profile results whose title contains the candidate name.
def ddg_results(html):
    s = BeautifulSoup(html, "html.parser")
    for r in s.select("div.result"):
        a = r.select_one("a.result__a")
        if not a:
            continue
        href = urllib.parse.parse_qs(urllib.parse.urlparse(a["href"]).query).get("uddg", [a["href"]])[0]
        sn = r.select_one(".result__snippet")
        yield href, a.get_text(" ", strip=True), sn.get_text(" ", strip=True) if sn else ""

hub_candidates = []
for rec in [r for r in log if r["stage"] == "ddg_search" and r["status"] == 200]:
    key = rec["candidate_key"]
    row = scope_by_key.loc[key]
    core = core_name(row["candidate_name"])
    own = county_variants(row["county_city"])
    for href, title, snippet in ddg_results(read_raw(rec)):
        n = normalize(href)
        if not n:
            continue
        label = account_label(title)
        profile = is_profile_url(href)
        # The account/page name itself must carry the candidate name; a post title that is
        # just post text mentioning the name (e.g. a news outlet's post) does not count.
        if profile and not (core in label and len(label) <= len(core) + 10):
            continue
        if not profile and not (label.startswith(core) and len(label) <= len(core) + 4):
            continue
        text = title + " " + snippet
        other = [c for c in COUNTIES if c != row["county_city"] and any(v in text for v in county_variants(c))]
        mentions_own = any(v in text for v in own)
        if any(w in label for w in FAN_WORDS):
            tier = "excluded_fan_or_media_page"
        elif other and not mentions_own:
            tier = "excluded_other_region"
        elif profile and (any(w in text for w in ROLE_WORDS) or mentions_own):
            tier = "uncertain_search_profile_role_match"
        else:
            # Posts: role words in a post usually describe other people, so name match only.
            tier = "uncertain_search_profile_name_only"
        obs.append(dict(candidate_key=key, platform=n[0], account_url=n[1], tier=tier, source_url=href,
                        source_file=rec["path"], source_sha256=rec["sha256"], source_title=title,
                        source_excerpt=snippet[:300], query=rec["query"]))

# 3. Link-hub pages (full page read programmatically).
for rec in [r for r in log if r["stage"] == "link_hub" and r["status"] == 200]:
    html = read_raw(rec)
    soup = BeautifulSoup(html, "html.parser")
    core = core_name(scope_by_key.loc[rec["candidate_key"], "candidate_name"])
    page_title = soup.title.get_text(" ", strip=True) if soup.title else ""
    if core not in page_title:
        continue
    links = set(re.findall(r'https?://(?:www\.|m\.)?(?:facebook|instagram|threads)\.(?:com|net)/[^"\'\s<>\\]+', html))
    for link in links:
        n = normalize(link)
        if n:
            obs.append(dict(candidate_key=rec["candidate_key"], platform=n[0], account_url=n[1],
                            tier="candidate_link_hub_listed", source_url=rec["url"], source_file=rec["path"],
                            source_sha256=rec["sha256"], source_title=page_title, source_excerpt="", query=""))

evidence = pd.DataFrame(obs).drop_duplicates(["candidate_key", "platform", "account_url", "source_url"])
evidence.insert(1, "candidate_name", evidence["candidate_key"].map(scope_by_key["candidate_name"]))
evidence.insert(2, "county_city", evidence["candidate_key"].map(scope_by_key["county_city"]))
evidence.insert(3, "office_type", evidence["candidate_key"].map(scope_by_key["office_type"]))
evidence["checked_date"] = "2026-11-16"
# Titles used for coding must literally occur in the saved raw file (entity-decoded text).
for rec_path, grp in evidence.groupby("source_file"):
    text = BeautifulSoup((PROJECT_ROOT / rec_path).read_bytes().decode("utf-8", "replace"), "html.parser").get_text(" ", strip=True)
    for t in grp["source_title"]:
        assert re.sub(r"\s+", "", t) in re.sub(r"\s+", "", text) or "串仔" in t or t == "", (rec_path, t)
print(evidence.groupby(["platform", "tier"]).size().unstack(fill_value=0).to_string())


串仔 pages matched to scope candidates: 62


tier       candidate_link_hub_listed  excluded_fan_or_media_page  excluded_other_region  third_party_directory_listed  uncertain_search_profile_name_only  uncertain_search_profile_role_match
platform                                                                                                                                                                                      
Facebook                          12                           8                      5                            70                                 196                                  150
Instagram                          8                           1                     11                            52                                 213                                   79
Threads                            3                           1                      5                            51                                  55                                   37


In [4]:
RANK = {"third_party_directory_listed": 1, "candidate_link_hub_listed": 2,
        "uncertain_search_profile_role_match": 3, "uncertain_search_profile_name_only": 4}
NOTE = {
    "third_party_directory_listed": "串仔（民進黨支持者整理，非官方）候選人頁列出；目標帳號未開啟核對",
    "candidate_link_hub_listed": "候選人姓名之 Linktree 類連結頁列出；連結頁歸屬依姓名判斷，目標帳號未開啟",
    "uncertain_search_profile_role_match": "不確定：僅搜尋結果摘要（平台帳號標題含姓名，摘要含參選／職務／縣市字詞），未讀帳號頁全文",
    "uncertain_search_profile_name_only": "不確定：僅帳號名稱與姓名相符，可能為同名他人，需人工確認",
}
usable = evidence[evidence["tier"].isin(RANK)].copy()
usable["rank"] = usable["tier"].map(RANK)

new = scope.copy()
for p in PLATFORMS:
    new[f"{p}_note"] = ""
filled = collections.Counter()
for p in PLATFORMS:
    for i, r in new.iterrows():
        cand = usable[(usable["candidate_key"] == r["candidate_key"]) & (usable["platform"] == p)]
        if r[f"{p}_status"] == "not_located":
            if cand.empty:
                continue
            best = cand["rank"].min()
            urls = list(dict.fromkeys(cand.loc[cand["rank"] == best, "account_url"]))
            tier = cand.loc[cand["rank"] == best, "tier"].iloc[0]
            if tier == "uncertain_search_profile_name_only" and len(urls) > 1:
                # Several same-name accounts with no candidacy wording: do not guess.
                new.at[i, f"{p}_status"] = "uncertain_same_name_multiple"
                new.at[i, f"{p}_note"] = f"不確定：搜尋到 {len(urls)} 個同名帳號，無參選字詞可判斷，未填入；見證據表"
                filled[(r["office_type"], p, "uncertain_same_name_multiple")] += 1
                continue
            new.at[i, f"{p}_urls"] = json.dumps(urls, ensure_ascii=False)
            new.at[i, f"{p}_status"] = tier
            note = NOTE[tier] + ("；同層級有多個帳號，需人工擇一" if len(urls) > 1 else "")
            new.at[i, f"{p}_note"] = note
            filled[(r["office_type"], p, tier)] += 1
        else:
            old = set(json.loads(r[f"{p}_urls"]))
            corroborated = cand[(cand["tier"] == "third_party_directory_listed") & cand["account_url"].isin(old)]
            if not corroborated.empty:
                new.at[i, f"{p}_note"] = "既有連結；本輪串仔（非官方）亦列出同一帳號"

# Existing non-missing cells must be unchanged.
for p in PLATFORMS:
    was = scope[f"{p}_status"] != "not_located"
    assert (new.loc[was, f"{p}_urls"] == scope.loc[was, f"{p}_urls"]).all()
    assert (new.loc[was, f"{p}_status"] == scope.loc[was, f"{p}_status"]).all()
assert list(new["candidate_key"]) == list(scope["candidate_key"])

rows = []
for office in sorted(new["office_type"].unique()):
    for p in PLATFORMS:
        b = scope[scope["office_type"] == office][f"{p}_status"]
        a = new[new["office_type"] == office][f"{p}_status"]
        rec = dict(office_type=office, platform=p, candidates=len(a),
                   missing_before=int((b == "not_located").sum()),
                   missing_after=int(a.isin(["not_located", "uncertain_same_name_multiple"]).sum()))
        for t in [*RANK, "uncertain_same_name_multiple"]:
            rec[f"filled_{t}"] = int((a == t).sum())
        rows.append(rec)
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print("total missing before:", int(summary["missing_before"].sum()), "after:", int(summary["missing_after"].sum()))
outputs = {"scope_new": new, "evidence": evidence, "summary": summary.astype(str)}


office_type  platform  candidates  missing_before  missing_after  filled_third_party_directory_listed  filled_candidate_link_hub_listed  filled_uncertain_search_profile_role_match  filled_uncertain_search_profile_name_only  filled_uncertain_same_name_multiple
      直轄市議員  Facebook         199              53             13                                    3                                 1                                          26                                         10                                    9
      直轄市議員 Instagram         199             112             45                                    2                                 2                                          43                                         20                                   23
      直轄市議員   Threads         199             137             87                                    2                                 1                                          28                                         

In [5]:
verification = {}
for key, frame in outputs.items():
    path = output_paths[key]
    mode = "w" if key == "scope_new" else "x"  # in-place update only for the authorized wide table
    with path.open(mode, encoding="utf-8-sig", newline="") as stream:
        frame.to_csv(stream, index=False)
    reread = pd.read_csv(path, **OUTPUT_READ_OPTIONS[key])
    assert list(reread.columns) == list(frame.columns), f"Schema changed while saving {key}."
    assert len(reread) == len(frame), f"Row count changed while saving {key}."
    pd.testing.assert_frame_equal(frame.reset_index(drop=True).astype(str), reread.astype(str), check_dtype=False)
    verification[key] = {"data_path": OUTPUTS[key], "rows": len(reread), "columns": len(reread.columns),
                         "sha256": sha256_file(path), "status": "read-back checks passed"}
    print(verification[key])
reread = pd.read_csv(output_paths["scope_new"], **OUTPUT_READ_OPTIONS["scope_new"])
assert reread["candidate_key"].is_unique and len(reread) == 280
for k in ("request_log", "manifest"):
    assert sha256_file(input_paths[k]) == input_fingerprints[k], "A raw input changed."


{'data_path': 'data/processed/candidate_social_links_scope_20261114.csv', 'rows': 280, 'columns': 13, 'sha256': 'd20a9aaafba91da223218505a544c571aa631461b285e364d351ff08c06424e1', 'status': 'read-back checks passed'}


{'data_path': 'data/processed/social_link_evidence_added_20261116.csv', 'rows': 957, 'columns': 14, 'sha256': 'a584a27d7a5d3c02d0efff9177fd9f681a1780b129b5e6e8a4501b1531c21f5b', 'status': 'read-back checks passed'}


{'data_path': 'data/processed/social_link_missing_summary_20261116.csv', 'rows': 9, 'columns': 10, 'sha256': '6000eb7a965f36f46962da4cf1fc8abf90a9f87d1e18db28dadbc89a5c4ae508', 'status': 'read-back checks passed'}


## 限制與後續

- 搜尋摘要層級（`uncertain_*`）都標為不確定：未開啟帳號頁全文，可能是同名他人、粉絲頁或已停用帳號。採用前請點開帳號，確認名稱、簡介的縣市與選區、參選身分。
- 串仔只收民進黨候選人，且是支持者整理的非官方工具；國民黨、民眾黨、無黨籍與小黨候選人只能靠搜尋與連結頁。
- `not_located` 仍代表本輪未找到，不代表候選人沒有帳號。
- 搜尋引擎結果會隨時間變動；本輪原始結果頁已保存，可重現本次判斷。
